# Schema Enforcement with GGUF Model Surgery

This notebook demonstrates applying and validating schemas for GGUF model parameters and tensors.

In [ ]:
import json
from pathlib import Path
from evolution.gguf.surgeon import ModelSurgeon
from evolution.gguf.grammar import SchemaValidator

## 1. Define Schema

Create a schema for model parameters and tensors:

In [ ]:
# Define schema
schema = {
    "name": "llama2.gguf",
    "version": "1.0",
    "parameters": {
        "type": "object",
        "properties": {
            "context_length": {"type": "integer", "minimum": 512, "maximum": 32768},
            "embedding_length": {"type": "integer", "minimum": 32, "maximum": 8192},
            "feed_forward_length": {"type": "integer", "minimum": 256, "maximum": 16384},
            "attention_heads": {"type": "integer", "minimum": 8, "maximum": 128},
            "vocab_size": {"type": "integer", "minimum": 1000, "maximum": 100000}
        },
        "required": ["context_length", "embedding_length", "attention_heads", "vocab_size"]
    },
    "tensors": {
        "type": "object",
        "properties": {
            "token_embeddings": {
                "type": "array", 
                "dimensions": 2,
                "dtype": "float16"
            },
            "output_norm": {
                "type": "array",
                "dimensions": 1,
                "dtype": "float32"
            }
        }
    }
}

# Save schema
schema_path = "schema.json"
with open(schema_path, "w") as f:
    json.dump(schema, f, indent=2)

## 2. Initialize Validator

Create validator and load schema:

In [ ]:
# Initialize validator
validator = SchemaValidator(schema_path)
surgeon = ModelSurgeon()

print("Schema loaded:")
print(f"Parameters: {len(schema['parameters']['properties'])}")
print(f"Tensors: {len(schema['tensors']['properties'])}")

## 3. Validate Model

Check model against schema:

In [ ]:
# Validate model
model_path = "model.gguf"
valid, errors = validator.validate(model_path)

if valid:
    print("✅ Model validates against schema")
else:
    print("❌ Schema validation failed:")
    for error in errors:
        print(f"- {error}")

## 4. Update Parameters

Fix any non-compliant parameters:

In [ ]:
# Apply schema-compliant updates
updates = {
    "context_length": 2048,
    "embedding_length": 4096,
    "attention_heads": 32
}

# Preview changes
preview = surgeon.preview(model_path, {
    "params": updates
})

print("Changes preview:")
print(f"Parameters to update: {len(updates)}")
print(f"Size delta: {preview.bytes_delta_mb:.1f}MB")

## 5. Apply Updates

Update model parameters to match schema:

In [ ]:
# Apply updates
out_path = "model.schema.gguf"
result = surgeon.update_parameters(
    model_path,
    updates,
    out_path
)

print("Parameter updates complete!")
print(f"Output model: {out_path}")

## 6. Revalidate

Check updated model against schema:

In [ ]:
# Revalidate
valid, errors = validator.validate(out_path)

if valid:
    print("✅ Updated model validates against schema")
    
    # Add provenance
    checksum, signature = surgeon.embed_provenance(
        out_path,
        {
            "type": "schema_update",
            "schema": schema["name"],
            "version": schema["version"]
        }
    )
    
    print(f"\nModel signed with checksum: {checksum}")
    print(f"Signature: {signature}")
else:
    print("❌ Schema validation still failing:")
    for error in errors:
        print(f"- {error}")